In [3]:
import os

In [4]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-'

In [8]:
from dataclasses import dataclass
from pathlib import Path

In [32]:
@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_col: str
    mlflow_url: str

In [33]:
from src.stroke_prediction.constants import *
from src.stroke_prediction.utils.common import read_yaml, create_directories, save_json

In [34]:
class ConfiguartionManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH): 
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root_dir])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.XGBoost
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_col = schema.name,
            mlflow_url = "https://dagshub.com/CarlosRoa76/Data-Science-Project-Stroke-Prediction-.mlflow"
        )
        return model_evaluation_config


In [38]:
import os
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow
import joblib
import mlflow.sklearn
import numpy as np
from urllib.parse import urlparse

In [41]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    
    def eval_metrics(self, actual, pred):
        acc = accuracy_score(actual, pred)
        prec = precision_score(actual, pred)
        rec = recall_score(actual, pred)
        f1 = f1_score(actual, pred)
        return acc, prec, rec, f1

    def log_into_mlflow(self):
        
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        X_test = test_data.drop([self.config.target_col], axis=1)
        y_test = test_data[[self.config.target_col]]

        mlflow.set_tracking_uri(self.config.mlflow_url)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(X_test)

            (acc, prec, rec, f1) = self.eval_metrics(y_test, predicted_qualities)
            
            #saving metrics
            scores = {
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "f1_score": f1
            }
            save_json(path=Path(self.config.metric_file_name), data=scores)

            # Log the model
            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("precision", prec)
            mlflow.log_metric("recall", rec)
            mlflow.log_metric("f1_score", f1)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="XGBoostStrokeModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [42]:
try:
    config = ConfiguartionManager().get_model_evaluation_config()
    model_eval = ModelEvaluation(config)
    model_eval.log_into_mlflow()
except Exception as e:
    raise e

2026-05-08 21:39:19,180 - INFO - JSON file saved at: artifacts\model_evaluation\metrics.json


2026/05/08 21:39:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 21:39:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoostStrokeModel'.
2026/05/08 21:39:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoostStrokeModel, version 1
Created version '1' of model 'XGBoostStrokeModel'.


🏃 View run wistful-mink-114 at: https://dagshub.com/CarlosRoa76/Data-Science-Project-Stroke-Prediction-.mlflow/#/experiments/0/runs/052c7f92e05d4767b314a8fb28332ca2
🧪 View experiment at: https://dagshub.com/CarlosRoa76/Data-Science-Project-Stroke-Prediction-.mlflow/#/experiments/0
